%md
# Agent Orchestration

Parts: 1. SQL path · 2. Vector path · 3. Router · 4. Synthesizer · 
5. Manual test · 6. LangGraph StateGraph · 7. Langfuse observability

## Setup — install dependencies and authenticate

In [0]:
%pip install langgraph langfuse
dbutils.library.restartPython()

In [0]:
%run ./setup/02_service_principal_auth

In [0]:
from typing import TypedDict, Optional
from openai import OpenAI
from langgraph.graph import StateGraph, END
from langfuse import Langfuse, observe
import requests
import time
import numpy as np

chat_client = OpenAI(
    api_key=sp_token,
    base_url=f"{WORKSPACE_URL}/ai-gateway/mlflow/v1"
)

embed_client = OpenAI(
    api_key=sp_token,
    base_url=f"{WORKSPACE_URL}/ai-gateway/mlflow/v1"
)

langfuse_client = Langfuse(
    public_key=dbutils.secrets.get(scope="olist-agent", key="langfuse-public-key"),
    secret_key=dbutils.secrets.get(scope="olist-agent", key="langfuse-secret-key"),
    host="https://cloud.langfuse.com"
)

WAREHOUSE_ID = "d45bde8408adc9ed"

%md
## Part 1: SQL path

Generates and executes SQL against the Gold layer to answer 
data/metric questions (e.g. "which state had the highest late 
delivery rate").

In [0]:
GOLD_SCHEMA_CONTEXT = """
Available tables (read-only, schema: gold):

1. gold.seller_performance
   - seller_id, seller_city, seller_state
   - total_orders, avg_review_score, avg_delivery_days, total_revenue
   - IMPORTANT: this table only includes sellers with at least one 
     delivered order. Sellers with zero delivered orders are ABSENT 
     from this table entirely.

2. gold.sales_by_category_month
   - category, month, total_orders, total_revenue, avg_item_price

3. gold.delivery_delay_analysis
   - order_id, customer_state, order_purchase_timestamp
   - order_estimated_delivery_date, order_delivered_customer_date
   - delay_days, delivery_status ('late' or 'on_time')

4. gold.seller_summary
   - total_sellers, sellers_with_delivered_orders, sellers_without_delivered_orders
   - Single-row table with pre-computed seller counts. Use this for 
     questions about sellers with/without delivered orders.
"""

In [0]:
chat_client = OpenAI(
    api_key=sp_token,
    base_url=f"{WORKSPACE_URL}/ai-gateway/mlflow/v1"
)

embed_client = OpenAI(
    api_key=sp_token,
    base_url=f"{WORKSPACE_URL}/ai-gateway/mlflow/v1"
)


def generate_sql(question: str) -> str:
    response = chat_client.chat.completions.create(
        model="system.ai.gpt-oss-120b",
        messages=[
            {"role": "system", "content": f"""You are a SQL generator. Given the schema below, write a single read-only PostgreSQL SELECT query that answers the user's question.

            If the available tables do NOT contain enough information to answer the question accurately, respond with exactly: NO_ANSWER
            Do not guess, fabricate values, or write a query that returns a placeholder/dummy result.

            Return ONLY the SQL query (or NO_ANSWER), no explanation, no markdown formatting.
            Always include a LIMIT clause (max 50 rows) unless the query is an aggregate returning a single row.

{GOLD_SCHEMA_CONTEXT}"""},
            {"role": "user", "content": question}
        ],
        max_tokens=1024
    )
    
    content = response.choices[0].message.content
    if isinstance(content, list):
        text = "".join(block.get("text", "") for block in content if isinstance(block, dict) and block.get("type") == "text")
    else:
        text = content
    
    return text.strip()



def validate_sql(sql: str) -> bool:
    normalized = sql.strip().upper()
    forbidden = ["INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "TRUNCATE", "CREATE", "GRANT"]
    if not normalized.startswith("SELECT"):
        return False
    if any(word in normalized for word in forbidden):
        return False
    if "LIMIT" not in normalized:
        return False
    return True

In [0]:
question = "Which state had the highest late delivery rate?"
sql = generate_sql(question)
print(sql)
print("Valid:", validate_sql(sql))

In [0]:
def execute_gold_query(sql: str, warehouse_id: str, timeout_seconds: int = 60) -> list:
    headers = {"Authorization": f"Bearer {sp_token}", "Content-Type": "application/json"}
    
    submit_response = requests.post(
        f"{WORKSPACE_URL}/api/2.0/sql/statements",
        headers=headers,
        json={
            "statement": sql,
            "warehouse_id": warehouse_id,
            "catalog": "olist_lakehouse",
            "schema": "gold"
        }
    )
    result = submit_response.json()
    statement_id = result["statement_id"]
    
    elapsed = 0
    while result["status"]["state"] in ("PENDING", "RUNNING"):
        if elapsed >= timeout_seconds:
            raise TimeoutError(f"Query exceeded {timeout_seconds}s (warehouse may be cold-starting)")
        time.sleep(1)
        elapsed += 1
        result = requests.get(
            f"{WORKSPACE_URL}/api/2.0/sql/statements/{statement_id}",
            headers=headers
        ).json()
    
    if result["status"]["state"] != "SUCCEEDED":
        raise Exception(f"Query failed: {result['status']}")
    
    return result.get("result", {}).get("data_array", [])

In [0]:
WAREHOUSE_ID = "d45bde8408adc9ed"

question = "Which state had the highest late delivery rate?"
sql = generate_sql(question)
print(f"Generated SQL:\n{sql}\n")

if validate_sql(sql):
    rows = execute_gold_query(sql, WAREHOUSE_ID)
    print(f"Result: {rows}")
else:
    print("Query rejected by security validation")

%md
## Debugging note: token expiration

While testing, the SQL execution call failed with a 403 "Invalid Token" 
error — the OAuth token generated via client-credentials has a short 
lifespan (appears to expire within a few minutes), and had expired 
during the warehouse's cold-start delay. This cell was used to inspect 
the raw API response and confirm the cause.

In [0]:
WAREHOUSE_ID = "d45bde8408adc9ed"

headers = {"Authorization": f"Bearer {sp_token}", "Content-Type": "application/json"}

submit_response = requests.post(
    f"{WORKSPACE_URL}/api/2.0/sql/statements",
    headers=headers,
    json={
        "statement": sql,
        "warehouse_id": WAREHOUSE_ID,
        "catalog": "olist_lakehouse",
        "schema": "gold"
    }
)

print("Status code:", submit_response.status_code)
print("Raw response:", submit_response.text)

%md
## Part 2: Vector search path

Embeds the question and searches the knowledge_base for semantically 
similar project documentation (e.g. "why did we get duplicate reviews").

In [0]:
def vector_search(question: str, top_k: int = 3) -> list:
    embedding_response = embed_client.embeddings.create(
        model="system.ai.qwen3-embedding-0-6b",
        input=question
    )
    question_embedding = embedding_response.data[0].embedding
    
    API_URL_BASE = "https://ep-twilight-snow-d88q9n3x.database.us-east-2.cloud.databricks.com/api/2.0/workspace/7474644702297592/rest/agent_knowledge_base"
    
    response = requests.post(
        f"{API_URL_BASE}/public/rpc/match_knowledge",
        headers={"Authorization": f"Bearer {sp_token}", "Content-Type": "application/json"},
        json={"query_embedding": question_embedding, "match_count": top_k}
    )
    return response.json()

In [0]:
results = vector_search("Why did order_reviews have duplicate records?")
print(results)

## Part 3: Router

Classifies the question into sql / vector / both, deciding which 
path(s) to run.

In [0]:
def route_question(question: str) -> str:
    response = chat_client.chat.completions.create(
        model="system.ai.gpt-oss-120b",
        messages=[
            {"role": "system", "content": """Classify the user's question into exactly one category:

            - "sql" — questions about metrics, numbers, aggregations, or business data 
            (e.g. revenue, delivery rates, seller performance, sales by category)
            - "vector" — questions about project history, data quality decisions, 
            or documentation (e.g. why something was fixed, what issues were found)
            - "both" — questions that need both a metric AND context about it

            Respond with ONLY one word: sql, vector, or both."""},
            {"role": "user", "content": question}
        ],
        max_tokens=10
    )
    
    content = response.choices[0].message.content
    if isinstance(content, list):
        content = "".join(b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text")
    
    route = content.strip().lower()
    return route if route in ("sql", "vector", "both") else "both"

## Part 4: Synthesizer

Combines the SQL result and/or vector search result into a final 
natural-language answer.

In [0]:
def synthesize_answer(question: str, sql_result=None, vector_result=None) -> str:
    context_parts = []
    
    if sql_result is not None:
        context_parts.append(f"Query result from the data warehouse: {sql_result}")
    
    if vector_result is not None:
        context_parts.append("Relevant project documentation:\n" + "\n".join(
            f"- {r['content']} (source: {r['source']})" for r in vector_result
        ))
    
    context = "\n\n".join(context_parts)
    
    response = chat_client.chat.completions.create(
        model="system.ai.gpt-oss-120b",
        messages=[
            {"role": "system", "content": """You are a data assistant for the Olist Lakehouse Pipeline project. 
            Answer the user's question clearly and concisely, based only on the provided context. 
            If the context includes a query result, state the number/fact directly. 
            If it includes documentation, explain the relevant finding."""},
            {"role": "user", "content": f"Question: {question}\n\nContext:\n{context}"}
        ],
        max_tokens=500
    )
    
    content = response.choices[0].message.content
    if isinstance(content, list):
        content = "".join(b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text")
    
    return content.strip()

## Part 5: Manual end-to-end test

Runs the full flow manually (router → conditional SQL/vector → 
synthesizer) before wrapping it as a formal LangGraph.

In [0]:
question = "Which state had the highest late delivery rate, and do we know why?"

route = route_question(question)
print(f"Route: {route}")

sql_result = None
vector_result = None

if route in ("sql", "both"):
    sql = generate_sql(question)
    if validate_sql(sql):
        sql_result = execute_gold_query(sql, WAREHOUSE_ID)

if route in ("vector", "both"):
    vector_result = vector_search(question)

answer = synthesize_answer(question, sql_result, vector_result)
print(f"\nAnswer: {answer}")

## Part 6: LangGraph — formal state graph

Wraps the router, SQL path, vector path, and synthesizer into a 
LangGraph StateGraph.

In [0]:
class AgentState(TypedDict):
    question: str
    route: Optional[str]
    sql_result: Optional[list]
    vector_result: Optional[list]
    answer: Optional[str]

In [0]:
def router_node(state: AgentState) -> dict:
    route = route_question(state["question"])
    return {"route": route}

def sql_node(state: AgentState) -> dict:
    sql = generate_sql(state["question"])
    if validate_sql(sql):
        result = execute_gold_query(sql, WAREHOUSE_ID)
        return {"sql_result": result}
    return {"sql_result": None}

def vector_node(state: AgentState) -> dict:
    result = vector_search(state["question"])
    return {"vector_result": result}

def synthesizer_node(state: AgentState) -> dict:
    answer = synthesize_answer(
        state["question"],
        state.get("sql_result"),
        state.get("vector_result")
    )
    return {"answer": answer}

In [0]:
def route_decision(state: AgentState) -> list:
    route = state["route"]
    if route == "sql":
        return ["sql_node"]
    elif route == "vector":
        return ["vector_node"]
    else:
        return ["sql_node", "vector_node"]

In [0]:
graph = StateGraph(AgentState)

graph.add_node("router_node", router_node)
graph.add_node("sql_node", sql_node)
graph.add_node("vector_node", vector_node)
graph.add_node("synthesizer_node", synthesizer_node)

graph.set_entry_point("router_node")
graph.add_conditional_edges("router_node", route_decision, ["sql_node", "vector_node"])
graph.add_edge("sql_node", "synthesizer_node")
graph.add_edge("vector_node", "synthesizer_node")
graph.add_edge("synthesizer_node", END)

agent = graph.compile()

In [0]:
result = agent.invoke({"question": "Which state had the highest late delivery rate, and do we know why?"})
print(result["answer"])

## Part 7: Langfuse observability

Instruments the LangGraph agent with tracing — capturing each node's 
execution, latency, and the LLM calls involved, for debugging and 
quality monitoring.

In [0]:
langfuse_client = Langfuse(
    public_key=dbutils.secrets.get(scope="olist-agent", key="langfuse-public-key"),
    secret_key=dbutils.secrets.get(scope="olist-agent", key="langfuse-secret-key"),
    host="https://cloud.langfuse.com"
)

In [0]:
langfuse_client = Langfuse(
    public_key=dbutils.secrets.get(scope="olist-agent", key="langfuse-public-key"),
    secret_key=dbutils.secrets.get(scope="olist-agent", key="langfuse-secret-key"),
    host="https://cloud.langfuse.com"
)


@observe(name="router")
def router_node(state: AgentState) -> dict:
    route = route_question(state["question"])
    return {"route": route}

@observe(name="sql_path")
def sql_node(state: AgentState) -> dict:
    sql = generate_sql(state["question"])
    if sql == "NO_ANSWER":
        return {"sql_result": None}
    if validate_sql(sql):
        result = execute_gold_query(sql, WAREHOUSE_ID)
        return {"sql_result": result}
    return {"sql_result": None}

@observe(name="vector_path")
def vector_node(state: AgentState) -> dict:
    result = vector_search(state["question"])
    return {"vector_result": result}

@observe(name="synthesizer")
def synthesizer_node(state: AgentState) -> dict:
    answer = synthesize_answer(
        state["question"],
        state.get("sql_result"),
        state.get("vector_result")
    )
    return {"answer": answer}

@observe(name="agent_run")
def run_agent(question: str) -> str:
    result = agent.invoke({"question": question})
    return result["answer"]

In [0]:
@observe(name="agent_run")
def run_agent(question: str) -> str:
    result = agent.invoke({"question": question})
    return result["answer"]

In [0]:
answer = run_agent("Which state had the highest late delivery rate, and do we know why?")
print(answer)

## Part 8: RAG evaluation

Small test set with reference answers, scored on semantic similarity 
(embedding-based) and LLM-as-judge (correctness, completeness, 
no hallucination).

In [0]:
eval_dataset = [
    {
        "question": "Which state had the highest late delivery rate?",
        "reference": "Alagoas (AL) had the highest late delivery rate, at approximately 23.9%."
    },
    {
        "question": "Why were there duplicate records in order_reviews?",
        "reference": "There were 814 duplicate review_id records after fixing a CSV parsing issue, caused by embedded commas and quotes in free-text review comments that shifted columns during ingestion."
    },
    {
        "question": "What was the total revenue for health_beauty category?",
        "reference": "The health_beauty category had a total revenue of approximately R$1,233,131.72, the highest among all categories."
    },
    {
        "question": "How many sellers have no delivered orders yet?",
        "reference": "125 out of 3,095 sellers (about 4%) have no delivered orders yet — their orders are still in other statuses."
    },
    {
        "question": "Why were some products marked as uncategorized?",
        "reference": "623 products were marked as uncategorized: 610 had no category in the source data (correlated with missing product photos), and 13 belonged to 2 categories missing from the translation table."
    },
]

In [0]:
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def semantic_similarity_score(answer: str, reference: str) -> float:
    answer_emb = embed_client.embeddings.create(model="system.ai.qwen3-embedding-0-6b", input=answer).data[0].embedding
    reference_emb = embed_client.embeddings.create(model="system.ai.qwen3-embedding-0-6b", input=reference).data[0].embedding
    return cosine_similarity(answer_emb, reference_emb)

In [0]:
def llm_judge_score(question: str, answer: str, reference: str) -> dict:
    response = chat_client.chat.completions.create(
        model="system.ai.gpt-oss-120b",
        messages=[
            {"role": "system", "content": """You are evaluating an AI agent's answer against a reference answer.
            Score the agent's answer from 0 to 10 on:
            - Correctness: does it match the key facts in the reference?
            - Completeness: does it cover what the reference covers?
            - No hallucination: does it avoid stating anything not supported by the reference or reasonable context?

            Respond in this exact format:
            score: <number 0-10>
            reasoning: <one sentence>"""},
            {"role": "user", "content": f"Question: {question}\n\nReference answer: {reference}\n\nAgent's answer: {answer}"}
        ],
        max_tokens=600
    )
    
    content = response.choices[0].message.content
    if isinstance(content, list):
        content = "".join(b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text")
    
    result = content.strip() if content else "ERROR: empty response (check token budget)"
    return result

In [0]:
eval_results = []

for item in eval_dataset:
    agent_answer = run_agent(item["question"])
    
    sim_score = semantic_similarity_score(agent_answer, item["reference"])
    judge_result = llm_judge_score(item["question"], agent_answer, item["reference"])
    
    eval_results.append({
        "question": item["question"],
        "agent_answer": agent_answer,
        "reference": item["reference"],
        "semantic_similarity": round(sim_score, 3),
        "llm_judge": judge_result
    })
    
    print(f"Q: {item['question']}")
    print(f"Similarity: {round(sim_score, 3)}")
    print(f"Judge: {judge_result}")
    print("-" * 80)

In [0]:
question = "How many sellers have no delivered orders yet?"
sql = generate_sql(question)
print(sql)